# CDDFuse Paper-Faithful MIF Training

120 ep / 2-phase / batch 8 / AMP fp16. Bám sát paper sect 5.2.

Inputs: `harvard-medical-train` (738) + `harvard-medical-fusion` (72) → merge thành full 810 pool.
Dataprocessing tự split 286 (130 train + 20 val + 136 test).

## Cell 1 — Config

In [ ]:
VARIANT     = 'Combined-Gated-Saliency'  # Module A.2 Gated × Module B.3 Saliency
REPO_URL    = 'https://github.com/kienvbhp872004/Image-Fusion.git'
REPO_BRANCH = 'main'
EPOCHS      = 120
EPOCH_GAP   = 40
BATCH       = 8
SEED        = 42
USE_AMP     = True

import os, subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = r.stdout.strip().splitlines()[0] if r.stdout else 'unknown'
print(f'[gpu] {GPU_NAME}')

## Cell 2 — Clone repo, install deps, downgrade torch cho P100 sm_60

In [ ]:
!git clone --branch $REPO_BRANCH $REPO_URL /kaggle/working/Image-Fusion
%cd /kaggle/working/Image-Fusion
!pip install -q einops==0.4.1 kornia==0.6.12 h5py tqdm scikit-image

# Downgrade torch để support P100 sm_60 (torch 2.6+ drop Pascal). 2.5.1 là bản cuối.
!pip uninstall -y torch torchvision torchaudio 2>&1 | tail -3
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!python -c "import torch; print('[torch]', torch.__version__, 'cuda:', torch.cuda.is_available(), 'cap:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'N/A')"

## Cell 3 — Stage Harvard medical full pool (merge 738+72 = 810)

In [ ]:
import shutil, pathlib, glob

def find_dataset_root(slug):
    for c in [f'/kaggle/input/{slug}', f'/kaggle/input/datasets/kienvbhp1234/{slug}']:
        if pathlib.Path(c).exists(): return pathlib.Path(c)
    m = glob.glob(f'/kaggle/input/**/{slug}', recursive=True)
    if m: return pathlib.Path(m[0])
    raise FileNotFoundError(slug)

TRAIN_SRC = find_dataset_root('harvard-medical-train')
TEST_SRC  = find_dataset_root('harvard-medical-fusion')
print(f'[paths] train={TRAIN_SRC}\n[paths] test ={TEST_SRC}')

# Stage thành cấu trúc dataprocessing_MIF.py mong đợi
POOL = pathlib.Path('/kaggle/working/Image-Fusion/Havard-Medical-Image-Fusion-Datasets-main/Havard-Medical-Image-Fusion-Datasets-main')
POOL.mkdir(parents=True, exist_ok=True)

for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    dst = POOL / modal
    dst.mkdir(parents=True, exist_ok=True)
    for src in [TRAIN_SRC / modal, TEST_SRC / modal]:
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    n_src = len(list((dst / modal.split('-')[0]).glob('*.png')))
    n_mri = len(list((dst / 'MRI').glob('*.png')))
    print(f'[stage] {modal}: src={n_src} mri={n_mri}')

## Cell 4 — Pre-process: extract patches → h5

In [ ]:
%cd /kaggle/working/Image-Fusion/models/MMIF-CDDFuse
!python dataprocessing_MIF.py

## Cell 5 — Train (120 ep, 2-phase, AMP fp16)

In [ ]:
amp_flag = '--amp' if USE_AMP else ''
!python train_MIF.py \
    --variant     $VARIANT \
    --num_epochs  $EPOCHS \
    --epoch_gap   $EPOCH_GAP \
    --batch       $BATCH \
    --seed        $SEED \
    --output      /kaggle/working/ \
    $amp_flag

## Cell 6 — Inference + per-image metrics

In [ ]:
import glob
ckpts = sorted(glob.glob(f'/kaggle/working/CDDFuse-{VARIANT}_MIF_*.pth'))
assert ckpts, 'No checkpoint found from Cell 5'
CKPT = ckpts[-1]
OUT_DIR = f'/kaggle/working/CDDFuse-{VARIANT}-PaperMIF'
print(f'[ckpt] {CKPT}')

# Stage test set theo paper split (từ MIF_split.json)
import json, shutil, pathlib
split = json.loads(open('/kaggle/working/Image-Fusion/models/MMIF-CDDFuse/data/MIF_split.json').read())
TEST_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/reference')
for modal, key in [('CT-MRI', 'test_CT'), ('PET-MRI', 'test_PET'), ('SPECT-MRI', 'test_SPECT')]:
    sub = modal.split('-')[0]
    (TEST_OUT / modal / sub).mkdir(parents=True, exist_ok=True)
    (TEST_OUT / modal / 'MRI').mkdir(parents=True, exist_ok=True)
    for pair in split[key]:
        shutil.copy(pair['src'], TEST_OUT / modal / sub / pathlib.Path(pair['src']).name)
        shutil.copy(pair['mri'], TEST_OUT / modal / 'MRI' / pathlib.Path(pair['mri']).name)
    print(f'[test ] {key}: {len(split[key])} pairs staged')

# Build variant_flag: skip --variant nếu là 'CDDFuse' baseline (không có trong eval registry)
variant_flag = f'--variant {VARIANT}' if VARIANT != 'CDDFuse' else ''
for modal in ['CT', 'PET', 'SPECT']:
    !python evaluate_cddfuse.py \
        $variant_flag \
        --modal         $modal \
        --ckpt          $CKPT \
        --harvard_root  /kaggle/working/Image-Fusion/data/reference \
        --out_dir       $OUT_DIR \
        --save_perimage

## Cell 7 — Package output

In [ ]:
import tarfile, json, hashlib, datetime, os, glob
h = hashlib.sha256()
with open(CKPT, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''): h.update(chunk)
stamp = {
    'variant_name': f'CDDFuse-{VARIANT}-PaperMIF',
    'based_on':     'CDDFuse paper sect 5.2',
    'datetime':     datetime.datetime.utcnow().isoformat() + 'Z',
    'train_mode':   'full_paper_120ep_2phase',
    'gpu':          GPU_NAME,
    'epochs':       EPOCHS,
    'epoch_gap':    EPOCH_GAP,
    'batch':        BATCH,
    'amp':          USE_AMP,
    'seed':         SEED,
    'ckpt_sha256':  h.hexdigest(),
}
with open(f'{OUT_DIR}/_ablation_stamp.json', 'w') as f:
    json.dump(stamp, f, indent=2)

tar_path = f'/kaggle/working/CDDFuse-{VARIANT}-PaperMIF_results.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(OUT_DIR, arcname=f'CDDFuse-{VARIANT}-PaperMIF')
    tar.add(CKPT, arcname=f'CDDFuse-{VARIANT}-PaperMIF.pth')
    # Include train_history.json (newly added in train_MIF.py)
    hist_files = glob.glob(f'/kaggle/working/CDDFuse-{VARIANT}_MIF_*_train_history.json')
    for hf in hist_files:
        tar.add(hf, arcname=os.path.basename(hf))
    split_json = '/kaggle/working/Image-Fusion/models/MMIF-CDDFuse/data/MIF_split.json'
    tar.add(split_json, arcname='MIF_split.json')
print('[done]', tar_path)